# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [6]:
import os

folder = r"C:\Users\berta\OneDrive\Desktop\IRONHACK"

print(os.listdir(folder))

['AIRBNB ANALYSIS', 'Class recordings', 'HANGMAN  Ppt.pptx', 'Panel de análisis de campañas de marketing.png', 'pokemon.csv', 'PROJECT 2', 'Project2PPT.pptx', 'Sharkattack_BNR (4).ipynb', 'Slide 1.docx', 'spaceship_titanic.csv', 'SQL1', 'TABLEAU', 'WEEK 1', 'we_fn_use_c_marketing_customer_value_analysis.csv', 'we_fn_use_c_marketing_customer_value_analysis.txt']


In [7]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\berta\OneDrive\Desktop\IRONHACK\spaceship_titanic.csv"
)

print(df.shape)
display(df.head())

(8693, 14)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["Transported", "PassengerId", "Name"])
y = df["Transported"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

print("Features before preprocessing:", X_train.shape[1])
print("Features after preprocessing:", X_train_scaled.shape[1])

Features before preprocessing: 11
Features after preprocessing: 5451


In [10]:
from sklearn.feature_selection import SelectKBest, f_classif

k = min(20, X_train_scaled.shape[1])

selector = SelectKBest(
    score_func=f_classif,
    k=k
)

X_train_selected = selector.fit_transform(
    X_train_scaled, y_train
)

X_test_selected = selector.transform(X_test_scaled)

print("Features before selection:", X_train_scaled.shape[1])
print("Features after selection:", X_train_selected.shape[1])

Features before selection: 5451
Features after selection: 20


In [12]:
feature_names = preprocessor.get_feature_names_out()
mask = selector.get_support()

feature_ranking = pd.DataFrame({
    "Feature": feature_names[mask],
    "ANOVA F-score": selector.scores_[mask]
}).sort_values("ANOVA F-score", ascending=False)

display(feature_ranking)

,Feature,ANOVA F-score
7,cat__CryoSleep_False,1892.287012
8,cat__CryoSleep_True,1892.287012
1,num__RoomService,444.933466
3,num__Spa,347.440197
4,num__VRDeck,301.474802
6,cat__HomePlanet_Europa,231.674912
5,cat__HomePlanet_Earth,202.435222
16,cat__Destination_55 Cancri e,73.740210
17,cat__Destination_TRAPPIST-1e,57.667384
0,num__Age,43.379824


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("selector", SelectKBest(score_func=f_classif)),
    ("model", RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

param_grid = {
    "selector__k": [10, 15, 20, "all"],
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best hyperparameters:")
print(grid_search.best_params_)

print(
    f"Best cross-validation accuracy: "
    f"{grid_search.best_score_:.2%}"
)

- Evaluate your model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("selector", SelectKBest(score_func=f_classif)),
    ("model", RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

param_grid = {
    "selector__k": [10, 15, 20, "all"],
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print(
    "Best cross-validation accuracy:",
    round(grid_search.best_score_ * 100, 2), "%"
)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(
    "Test accuracy:",
    round(accuracy_score(y_test, y_pred) * 100, 2), "%"
)

print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Transported", "Transported"],
    zero_division=0
))

Fitting 5 folds for each of 48 candidates, totalling 240 fits


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
#your code here

- Run Grid Search